# Spain's economy by prime minister: rebuilding (and stress-testing) a news graphic

A Spanish news outlet published an Instagram carousel comparing Pedro Sánchez (2018 to 2026) with his three predecessors, Aznar, Zapatero and Rajoy, on a dozen economic indicators. Its headline claims:

1. GDP grew **15.1%** under Sánchez, well above Rajoy (10.3%) and Zapatero (7.1%).
2. The minimum wage (SMI) rose about **66%**, far ahead of inflation.
3. Employment rose **18 to 19%** and unemployment fell to its lowest since 2008.
4. Housing is the weak spot: price per m² up **48.3%**, versus a **6.7%** fall under Rajoy.

In this notebook we:

* **Source** the underlying data ourselves from Eurostat's public API.
* **Reproduce** the article's comparison table.
* **Stress-test** it with four analytical moves the article mostly skips: annualizing, going per capita, separating nominal from real, and benchmarking against the euro area.

The goal is not to decide who was the better prime minister. It is to see how much a chart's *framing choices* shape the story told by the same numbers.

## 1. Sourcing the data

Everything comes from [Eurostat](https://ec.europa.eu/eurostat/web/main/data/database), which republishes harmonized national statistics from Spain's INE, Banco de España and Social Security. Using one source keeps definitions consistent across 30 years and lets us pull the euro area as a comparison group for free.

| Indicator | Eurostat dataset | Notes |
|---|---|---|
| Real GDP | `nama_10_gdp` | Chain-linked volumes, 2010 euros |
| Real GDP per capita | `nama_10_pc` | Chain-linked, euros per inhabitant |
| Prices (inflation) | `prc_hicp_aind` | HICP annual average index |
| Minimum wage | `earn_mw_cur` | Monthly, euros, semiannual; from 1999 |
| House prices | `prc_hpi_a` | House price index; Spain from 2007 |
| Employment | `nama_10_pe` | Persons employed, national accounts |
| Unemployment rate | `une_rt_a` | Age 15 to 74, % of labour force |
| Public debt | `gov_10dd_edpt1` | General government, % of GDP |
| Population | `demo_pjan` | 1 January |

**Differences from the article.** The article uses Spanish sources directly: the EPA survey and Social Security affiliates for jobs, the ministry's €/m² valuation series for housing, and the legal SMI table. Our versions are close cousins, not identical, so expect our numbers to differ by a few points. Two gaps matter most: Eurostat's minimum wage series starts in 1999 (so Aznar's term is partial), and its house price index for Spain starts in 2007 (so Aznar has no housing figure).

In [ ]:
import json
import urllib.parse
import urllib.request

import matplotlib.pyplot as plt
import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)

BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"


def eurostat(dataset, **filters):
    # Fetch a Eurostat dataset (JSON-stat 2.0) and return a tidy Polars DataFrame,
    # one row per observation, one column per dimension plus 'value'.
    params = {"lang": "EN", "format": "JSON", **filters}
    url = BASE + dataset + "?" + urllib.parse.urlencode(params, doseq=True)
    with urllib.request.urlopen(url, timeout=120) as resp:
        js = json.load(resp)

    dims, sizes = js["id"], js["size"]
    # For each dimension, a list mapping position -> category code
    codes = []
    for d in dims:
        idx = js["dimension"][d]["category"]["index"]
        if isinstance(idx, list):
            codes.append(idx)
        else:
            inv = [None] * len(idx)
            for code_, pos in idx.items():
                inv[pos] = code_
            codes.append(inv)

    values = js["value"]
    items = values.items() if isinstance(values, dict) else enumerate(values)
    rows = []
    for flat, v in items:
        if v is None:
            continue
        flat = int(flat)
        rec = {}
        # JSON-stat stores the cube in row-major order: peel off the last dimension first
        for d, size, inv in reversed(list(zip(dims, sizes, codes))):
            flat, pos = divmod(flat, size)
            rec[d] = inv[pos]
        rec["value"] = float(v)
        rows.append(rec)
    return pl.DataFrame(rows).select(dims + ["value"])

Each request below names a dataset plus the filters that pin down a single series. If Eurostat renames a code (it occasionally restructures datasets), the request fails loudly and you only need to edit that one line.

In [ ]:
REQUESTS = {
    "gdp":    ("nama_10_gdp",    dict(geo=["ES", "EA20"], unit="CLV10_MEUR", na_item="B1GQ")),
    "gdp_pc": ("nama_10_pc",     dict(geo=["ES", "EA20"], unit="CLV10_EUR_HAB", na_item="B1GQ")),
    "hicp":   ("prc_hicp_aind",  dict(geo="ES", unit="INX_A_AVG", coicop="CP00")),
    "smi":    ("earn_mw_cur",    dict(geo="ES", currency="EUR")),
    "hpi":    ("prc_hpi_a",      dict(geo="ES", unit="I15_A_AVG", purchase="TOTAL")),
    "emp":    ("nama_10_pe",     dict(geo="ES", unit="THS_PER", na_item="EMP_DC")),
    "unemp":  ("une_rt_a",       dict(geo="ES", age="Y15-74", sex="T", unit="PC_ACT")),
    "debt":   ("gov_10dd_edpt1", dict(geo="ES", sector="S13", na_item="GD", unit="PC_GDP")),
    "pop":    ("demo_pjan",      dict(geo="ES", age="TOTAL", sex="T", unit="NR")),
}

raw = {}
for name, (ds, f) in REQUESTS.items():
    try:
        raw[name] = eurostat(ds, **f)
        print(f"{name:7s} {ds:15s} {raw[name].height:4d} rows")
    except Exception as e:
        print(f"{name:7s} {ds:15s} FAILED: {e}")

### Building one annual table

The raw pulls have different time formats (`2024`, `2024-S1`) and some include the euro area. We reduce each to a `(year, value)` series and join them into a single wide table. For the semiannual minimum wage we keep the first half of each year, since Spain sets the SMI in January.

In [ ]:
def series(name, col, geo="ES", period_suffix=None):
    df = raw[name].filter(pl.col("geo") == geo)
    if period_suffix:
        df = df.filter(pl.col("time").str.ends_with(period_suffix))
    return df.select(
        pl.col("time").str.slice(0, 4).cast(pl.Int32).alias("year"),
        pl.col("value").alias(col),
    ).unique("year")


parts = [
    series("gdp", "gdp"),
    series("gdp", "gdp_ea", geo="EA20"),
    series("gdp_pc", "gdp_pc"),
    series("gdp_pc", "gdp_pc_ea", geo="EA20"),
    series("hicp", "hicp"),
    series("smi", "smi", period_suffix="S1"),
    series("hpi", "hpi"),
    series("emp", "emp"),
    series("unemp", "unemp"),
    series("debt", "debt"),
    series("pop", "pop"),
]

es = pl.DataFrame({"year": pl.int_range(1995, 2027, eager=True).cast(pl.Int32)})
for p in parts:
    es = es.join(p, on="year", how="left")

# Real (inflation-adjusted) versions: divide by the price index
es = es.with_columns(
    (pl.col("smi") / pl.col("hicp") * 100).alias("smi_real"),
    (pl.col("hpi") / pl.col("hicp") * 100).alias("hpi_real"),
)
es

### Defining the terms

We approximate each term with calendar years, as the article's annual charts do. Sánchez took office in June 2018, so 2018 counts as both Rajoy's end year and Sánchez's start year. Sánchez's end year is the latest year with GDP data.

In [ ]:
LAST = es.filter(pl.col("gdp").is_not_null())["year"].max()

TERMS = [
    ("Aznar (PP)",      1996, 2004),
    ("Zapatero (PSOE)", 2004, 2011),
    ("Rajoy (PP)",      2011, 2018),
    ("Sánchez (PSOE)",  2018, LAST),
]
print("Latest full year of GDP data:", LAST)

## 2. Reproducing the article's table

First, the article's own numbers, transcribed from the graphic (using the 2018 to 2026 column where it exists). Then ours, computed the same way: percent change from the first year to the last year of each term, or the change in percentage points for rates.

### Original Instagram graphics

The comparison table and accompanying charts are included below for reference. The images are loaded from GitHub so they also appear in Colab; the `ig-imagenes` folder must be published on the repository's `main` branch.

<table>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/Colores.jpeg" alt="Original comparison table" width="360"><br>Comparison table</td>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/PIB.jpeg" alt="Minimum wage and inflation chart" width="360"><br>Minimum wage and inflation</td>
  </tr>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/PIB_cuartos.jpeg" alt="GDP growth chart" width="360"><br>GDP growth</td>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/Vivienda.jpeg" alt="Housing prices and wages chart" width="360"><br>Housing</td>
  </tr>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/migracion.jpeg" alt="Employment chart" width="360"><br>Employment</td>
    <td></td>
  </tr>
</table>

In [ ]:
article = pl.DataFrame(
    {
        "indicator": ["Real GDP", "Real GDP per capita", "Inflation", "Minimum wage (nominal)",
                      "Minimum wage (real)", "House prices (nominal)", "House prices (real)",
                      "Employment", "Unemployment rate (pp)", "Public debt (pp of GDP)"],
        "Aznar (PP)":      [34.5, 25.2, 25.1, 18.0, -5.7, 120.5, 76.3, 41.3, -9.0, -20.1],
        "Zapatero (PSOE)": [7.1, -1.3, 19.9, 39.3, 16.2, 10.6, -7.8, 1.5, 10.4, 24.2],
        "Rajoy (PP)":      [10.3, 9.7, 6.9, 14.7, 7.3, -6.7, -12.7, 4.9, -6.1, 30.3],
        "Sánchez (PSOE)":  [15.1, 6.2, 26.5, 65.9, 31.2, 48.3, 17.2, 17.9, -5.4, 1.8],
    }
)
article

In [ ]:
INDICATORS = [
    # (label, column, kind)  kind: "pct" = % change, "pp" = change in points
    ("Real GDP", "gdp", "pct"),
    ("Real GDP per capita", "gdp_pc", "pct"),
    ("Inflation", "hicp", "pct"),
    ("Minimum wage (nominal)", "smi", "pct"),
    ("Minimum wage (real)", "smi_real", "pct"),
    ("House prices (nominal)", "hpi", "pct"),
    ("House prices (real)", "hpi_real", "pct"),
    ("Employment", "emp", "pct"),
    ("Unemployment rate (pp)", "unemp", "pp"),
    ("Public debt (pp of GDP)", "debt", "pp"),
    ("Population", "pop", "pct"),
]


def value(col, year):
    v = es.filter(pl.col("year") == year)[col]
    return v[0] if len(v) else None


def change(col, start, end, kind="pct", annualize=False):
    a, b = value(col, start), value(col, end)
    if a is None or b is None:
        return None
    years = end - start
    if kind == "pp":
        return (b - a) / years if annualize else b - a
    if annualize:
        return ((b / a) ** (1 / years) - 1) * 100
    return (b / a - 1) * 100


def term_table(annualize=False):
    data = {"indicator": [lab for lab, _, _ in INDICATORS]}
    for term, s, e in TERMS:
        data[term] = [change(col, s, e, kind, annualize) for _, col, kind in INDICATORS]
    return pl.DataFrame(data).with_columns(pl.exclude("indicator").round(1))


ours = term_table()
ours

**Check:** put the two side by side. Where do we match closely, and where do differences in source (EPA vs national accounts employment, €/m² valuations vs a price index) or missing years show up?

In [ ]:
terms = [t for t, _, _ in TERMS]
side = (
    article.join(ours, on="indicator", suffix=" [ours]")
    .select(["indicator"] + [c for t in terms for c in (t, f"{t} [ours]")])
)
side

## 3. Reproducing the article's charts

In [ ]:
COLORS = {"Aznar (PP)": "#9aa7b8", "Zapatero (PSOE)": "#c9625b",
          "Rajoy (PP)": "#4a6fa5", "Sánchez (PSOE)": "#c0392b"}


def shade_terms(ax, y=None):
    for i, (term, s, e) in enumerate(TERMS):
        if i % 2 == 0:
            ax.axvspan(s, e, color="0.93", zorder=0)
        ax.text((s + e) / 2, ax.get_ylim()[1] if y is None else y, term.split(" (")[0],
                ha="center", va="top", fontsize=9, style="italic")

In [ ]:
# Article slide 5: annual real GDP growth
g = es.select("year", (pl.col("gdp").pct_change() * 100).alias("growth")).drop_nulls()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(g["year"], g["growth"],
       color=["#2c5aa0" if v >= 0 else "#c0392b" for v in g["growth"]])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Real GDP growth, % y/y")
ax.set_title("How GDP did in each term")
shade_terms(ax)
plt.show()

In [ ]:
# Article slide 3: minimum wage vs prices, indexed to the first year of SMI data
base = es.filter(pl.col("smi").is_not_null())["year"].min()
idx = es.select(
    "year",
    (pl.col("smi") / value("smi", base) * 100).alias("Minimum wage"),
    (pl.col("hicp") / value("hicp", base) * 100).alias("Prices (HICP)"),
).filter(pl.col("year") >= base)

fig, ax = plt.subplots(figsize=(10, 4))
for col, c in [("Minimum wage", "#2c5aa0"), ("Prices (HICP)", "#c0392b")]:
    ax.plot(idx["year"], idx[col], marker="o", ms=3, label=col, color=c)
ax.set_title(f"Minimum wage vs inflation, {base} = 100")
ax.legend()
shade_terms(ax)
plt.show()

In [ ]:
# Article slide 4: employment
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(es["year"], es["emp"] / 1000, color="#8fb0de")
ax.set_ylim(0, None)
ax.set_ylabel("Employed persons, millions")
ax.set_title("Employment (national accounts)")
shade_terms(ax)
plt.show()

latest_u = value("unemp", LAST)
prior = es.filter((pl.col("year") < LAST) & (pl.col("unemp") <= latest_u))["year"].max()
print(f"Unemployment in {LAST}: {latest_u:.1f}%. Last year at or below that level: {prior}")

**Check the claim.** The article says unemployment is at its lowest since 2008. Does the printed year match? Also notice what "lowest since 2008" leaves unsaid: how high is the rate in absolute terms? We come back to this with the euro-area comparison.

## 4. Stress test 1: annualize

Terms have different lengths. A cumulative percentage rewards a longer term even at the same pace. Converting to a compound annual rate puts every term on the same clock:

$$\text{annual rate} = \left(\frac{x_\text{end}}{x_\text{start}}\right)^{1/\text{years}} - 1$$

For rates measured in points (unemployment, debt) we divide the change by the number of years.

In [ ]:
lengths = pl.DataFrame({"term": [t for t, _, _ in TERMS], "years": [e - s for _, s, e in TERMS]})
print(lengths)
annual = term_table(annualize=True)
annual

**Question.** The GDP slide compares Sánchez with Rajoy and Zapatero but not Aznar. Using the annualized table, where does each term rank on real GDP growth? Does the ranking change compared with the cumulative table?

## 5. Stress test 2: go per capita

Spain's population grew substantially over this period, much of it through immigration. More people means more output even if the average person is no better off. Real GDP *per capita* is the better measure of living standards.

In [ ]:
pc = annual.filter(pl.col("indicator").is_in(["Real GDP", "Real GDP per capita", "Population"]))
print(pc)

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.38
x = range(len(TERMS))
tot = [change("gdp", s, e, annualize=True) for _, s, e in TERMS]
per = [change("gdp_pc", s, e, annualize=True) for _, s, e in TERMS]
ax.bar([i - w / 2 for i in x], tot, w, label="Total real GDP", color="#2c5aa0")
ax.bar([i + w / 2 for i in x], per, w, label="Real GDP per capita", color="#8fb0de")
ax.set_xticks(list(x), [t for t, _, _ in TERMS])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("% per year")
ax.set_title("Annualized growth: total vs per person")
ax.legend()
plt.show()

**Question.** How much of each term's GDP growth came from more people rather than more output per person? Which term's ranking moves the most when you switch measures? Notice that the article's own table contains the per capita row, but the GDP headline uses the total.

## 6. Stress test 3: nominal vs real

The minimum wage and housing headlines use nominal figures (66%, 48.3%), while the real figures sit in the table. Over a term with cumulative inflation in the 20s, nominal and real tell quite different stories.

In [ ]:
nr = ours.filter(pl.col("indicator").str.contains("Minimum wage|House prices|Inflation"))
nr

**Question.** The housing slide contrasts +48.3% under Sánchez with −6.7% under Rajoy. Rajoy's term began in 2011, just after the housing bubble burst. How much of each figure is the policy of the government in office and how much is the phase of the housing cycle it inherited? What would a fair comparison look like?

## 7. Stress test 4: benchmark against the euro area

Every term coincided with a shared shock: the euro boom (Aznar), the Great Recession (Zapatero), the euro debt crisis and recovery (Rajoy), COVID and the energy shock (Sánchez). One way to separate a government's performance from the global weather is to compare Spain with its peers over the same years.

In [ ]:
bench = pl.DataFrame({
    "term": [t for t, _, _ in TERMS],
    "Spain GDP pc %/yr": [change("gdp_pc", s, e, annualize=True) for _, s, e in TERMS],
    "Euro area GDP pc %/yr": [change("gdp_pc_ea", s, e, annualize=True) for _, s, e in TERMS],
}).with_columns(
    (pl.col("Spain GDP pc %/yr") - pl.col("Euro area GDP pc %/yr")).alias("Spain minus EA")
).with_columns(pl.exclude("term").round(2))
bench

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(bench["term"], bench["Spain minus EA"],
       color=["#2c5aa0" if v >= 0 else "#c0392b" for v in bench["Spain minus EA"]])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("percentage points per year")
ax.set_title("Spain's real GDP per capita growth relative to the euro area")
plt.show()

**Caveat.** The euro area series uses today's 20-member composition for every year, and Spain is itself part of it (roughly a tenth of euro-area GDP), which mutes the gap slightly.

## 8. What the slides leave out

The carousel turns four indicators into headline slides. Two rows of its own table get no slide:

In [ ]:
ours.filter(pl.col("indicator").str.contains("Inflation|Public debt"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(es["year"], es["debt"], color="#2c5aa0")
axes[0].set_title("Public debt, % of GDP")
axes[1].plot(es["year"], es["unemp"], color="#c0392b")
axes[1].set_title("Unemployment rate, %")
for ax in axes:
    shade_terms(ax)
plt.tight_layout()
plt.show()

A change of +1.8 points of GDP in debt sounds like stability, and given COVID spending it arguably is. But the *level* it stabilized at matters for the next government. The same goes for unemployment: a multi-year low can still be high relative to peers.

## 9. Discussion

1. Rank the four terms on real GDP growth three ways: cumulative, annualized, and annualized per capita. Which framing would each party's press office choose?
2. The article picked calendar endpoints. Re-run the table with Sánchez starting in 2019 (his first full year). How sensitive are the results to a one-year shift? Edit `TERMS` and re-run.
3. Pick one headline from the carousel and rewrite it so that it is equally short but harder to mislead with.
4. Which single indicator do you think best captures "how people's lives went" during a term? What does it miss?